In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
import pandas as pd

# ==========================================
# 1. 模拟上帝视角：构建真实的 Criteo 数据大盘
# ==========================================
np.random.seed(42)
N = 100000  # 模拟 10 万个 T=0 的用户

# 设定阶级：Top 10% 是 CO (爱点击)，剩下 90% 是 NT (不点击)
is_co = np.zeros(N)
is_co[:10000] = 1

# 模拟输入特征 1：E_c (因果类型 Embedding，Dense 特征)
# CO 的 E_c 特征极强 (均值 0.9)，NT 的特征极弱 (均值 0.1)
E_c = np.where(is_co == 1, np.random.normal(0.9, 0.05, N), np.random.normal(0.1, 0.05, N))

# 模拟输入特征 2：X (真实的稀疏购买力特征)
# 假设只有很少一部分人身上有真实的购买力特征 (X=1 代表土豪)
X_sparse = np.zeros(N)
# 设定真实转化率 (匹配你的真实数据)：CO 是 0.017, NT 是 0.002
X_sparse[:10000][np.random.rand(10000) < 0.017] = 1
X_sparse[10000:][np.random.rand(90000) < 0.002] = 1

# 真实标签 Y 就是购买力 (极其稀疏)
Y_true = X_sparse 

# ==========================================
# 2. 模拟模型训练：V1 (清醒) vs V2 (被污染)
# ==========================================

# 【V1 模型】：老实本分，只看原始特征 X
# (相当于隔离了 C 任务的纯 Y 塔)
features_v1 = X_sparse.reshape(-1, 1)
model_v1 = LogisticRegression(penalty='l2', C=1.0)
model_v1.fit(features_v1, Y_true)
pred_v1 = model_v1.predict_proba(features_v1)[:, 1]

# 【V2 模型】：早期拼接，强行混入 E_c
# (模拟 Shared Bottom 被高频 Dense 特征冲击)
# 注意：我们加大了 L2 正则 (C=0.01) 来模拟深度神经网络在面临海量稀疏 0 时产生的权重压缩
features_v2 = np.column_stack((X_sparse, E_c))
model_v2 = LogisticRegression(penalty='l2', C=0.01) 
model_v2.fit(features_v2, Y_true)
pred_v2 = model_v2.predict_proba(features_v2)[:, 1]

# ==========================================
# 3. 验收结果：出具对账单
# ==========================================
df = pd.DataFrame({
    'Group': np.where(is_co == 1, 'Top 10% (强 CO)', 'Bottom 90% (强 NT)'),
    'Real_y0': Y_true,
    'V1_Pred': pred_v1,
    'V2_Pred': pred_v2
})

result = df.groupby('Group').mean()[['Real_y0', 'V1_Pred', 'V2_Pred']].reset_index()

print("\n" + "="*80)
print("🔬 沙盒运行结果：为什么 V2 算出的 CO 反而比 NT 低？")
print("="*80)
print(result.to_string(index=False))
print("="*80)

# 打印 V2 学到的“畸形权重”
w_X, w_Ec = model_v2.coef_[0]
bias = model_v2.intercept_[0]
print(f"\n🧠 扒开 V2 的脑子看权重：")
print(f"全局偏置 (Bias): {bias:.4f}  <-- 这是发给 NT 的大锅饭")
print(f"原始购买特征权重 (w_X): {w_X:.4f}  <-- 被压扁了")
print(f"点击特征权重 (w_Ec): {w_Ec:.4f}  <-- 这是射向 CO 的子弹！")


🔬 沙盒运行结果：为什么 V2 算出的 CO 反而比 NT 低？
            Group  Real_y0  V1_Pred  V2_Pred
Bottom 90% (强 NT)   0.0020 0.002058 0.003135
   Top 10% (强 CO)   0.0189 0.017991 0.008461

🧠 扒开 V2 的脑子看权重：
全局偏置 (Bias): -5.8964  <-- 这是发给 NT 的大锅饭
原始购买特征权重 (w_X): 3.3021  <-- 被压扁了
点击特征权重 (w_Ec): 0.8621  <-- 这是射向 CO 的子弹！


In [2]:
import torch

# 1. 直接读取你 NAS 上的真实权重文件
ckpt_path = "/NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v2_emb/run_v2_emb/best_model.pth"

print(f"🔍 正在加载 V2 权重: {ckpt_path}")
try:
    state_dict = torch.load(ckpt_path, map_location='cpu')
except Exception as e:
    print(f"❌ 加载失败，请检查路径: {e}")
    exit()

# 兼容不同的保存格式 (有时候可能套了一层 'model_state_dict')
if 'model_state_dict' in state_dict:
    state_dict = state_dict['model_state_dict']

# 2. 提取 C 任务传过来的三个先验 Embedding
try:
    emb_never = state_dict['emb_never']   # shape: [c_embedding_dim]
    emb_comp = state_dict['emb_comp']     # shape: [c_embedding_dim]
    emb_always = state_dict['emb_always'] # shape: [c_embedding_dim]
    c_emb_dim = emb_never.shape[0]
except KeyError:
    print("❌ 找不到 emb_never/comp，你确定这是一个 V2 (joint_emb) 的模型吗？")
    exit()

print("\n" + "="*60)
print(f"🧬 1. 提取到的因果 Embedding (维度={c_emb_dim})")
print(f"  - NT (Never-taker) Embedding : {emb_never.numpy().round(4)}")
print(f"  - CO (Complier) Embedding   : {emb_comp.numpy().round(4)}")

# 3. 提取 Shared Bottom 第一层的权重矩阵
# V2 中，x_aug = torch.cat([x_main, c_feat], dim=1)
# 所以 shared_layers.0.weight 的最后 c_emb_dim 列，就是专门用来处理这些 Embedding 的！
layer0_weight = state_dict['shared_layers.0.weight'] # shape: [hidden_dim, input_dim + c_emb_dim]
layer0_bias = state_dict['shared_layers.0.bias']

# 把专门负责“对接” C Embedding 的权重切片切出来
# W_c 的 shape: [hidden_dim, c_emb_dim]
W_c = layer0_weight[:, -c_emb_dim:] 

# 4. 算账时间：看看 NT 和 CO 到底给底层网络注入了什么信号？
# 数学原理：矩阵乘法。我们看看 E_nt 和 E_co 经过 W_c 映射后，
# 给第一层隐藏层 (Hidden Layer 1) 带来了多大的数值偏移？

# shape: [hidden_dim]
impact_nt = torch.matmul(W_c, emb_never)
impact_co = torch.matmul(W_c, emb_comp)

print("\n" + "="*60)
print("⚔️ 2. 底层神经元冲击波测试 (Shared Bottom 第一层)")
print("解释：正数代表提升基准分，负数代表压制基准分")

print(f"\n[NT 群体的冲击波]")
print(f"  - 向量均值: {impact_nt.mean().item():.4f}")
print(f"  - 负向神经元个数: {(impact_nt < 0).sum().item()} / {len(impact_nt)}")
print(f"  - 冲击波前 5 个神经元的值: {impact_nt[:5].numpy().round(4)}")

print(f"\n[CO 群体的冲击波]")
print(f"  - 向量均值: {impact_co.mean().item():.4f}")
print(f"  - 负向神经元个数: {(impact_co < 0).sum().item()} / {len(impact_co)}")
print(f"  - 冲击波前 5 个神经元的值: {impact_co[:5].numpy().round(4)}")

print("\n" + "="*60)
print("💡 终极审判：")
diff = impact_co.mean() - impact_nt.mean()
if diff < 0:
    print(f"🚨 实锤了！CO 的 Embedding 被底层网络打上了严重的负向烙印！")
    print(f"🚨 CO 的平均神经元冲击比 NT 低了 {abs(diff):.4f}。这就是 CO(0.0047) < NT(0.0069) 的根本原因！")
else:
    print("👀 咦？第一层看来没背锅，压制可能发生在更深层或者 y0 专属 head 里，我们需要看全量前向传播。")

🔍 正在加载 V2 权重: /NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v2_emb/run_v2_emb/best_model.pth

🧬 1. 提取到的因果 Embedding (维度=16)
  - NT (Never-taker) Embedding : [-2.7186 -0.0489 -0.682   1.6377  0.408  -1.2775 -1.1105  3.3162 -0.8403
 -0.6311 -0.8403  0.1627 -0.477   0.1451 -1.69    1.5412]
  - CO (Complier) Embedding   : [ 0.5499  1.2452 -0.5264  1.3729  1.1289  2.0578  0.2094  0.412  -0.4758
 -0.7418  0.1483  0.3389  0.3963 -0.386   0.8018 -1.7842]

⚔️ 2. 底层神经元冲击波测试 (Shared Bottom 第一层)
解释：正数代表提升基准分，负数代表压制基准分

[NT 群体的冲击波]
  - 向量均值: -0.0221
  - 负向神经元个数: 35 / 64
  - 冲击波前 5 个神经元的值: [ 0.5981  0.0795 -0.2741  0.2357 -0.6122]

[CO 群体的冲击波]
  - 向量均值: 0.0773
  - 负向神经元个数: 28 / 64
  - 冲击波前 5 个神经元的值: [-0.0657  0.1296  0.0802 -0.3959  0.8576]

💡 终极审判：
👀 咦？第一层看来没背锅，压制可能发生在更深层或者 y0 专属 head 里，我们需要看全量前向传播。


In [3]:
import torch
import torch.nn.functional as F

# 1. 挂载真实权重
ckpt_path = "/NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v2_emb/run_v2_emb/best_model.pth"
state_dict = torch.load(ckpt_path, map_location='cpu')
if 'model_state_dict' in state_dict:
    state_dict = state_dict['model_state_dict']

# 2. 提取 Embedding
emb_never = state_dict['emb_never']
emb_comp = state_dict['emb_comp']
c_emb_dim = emb_never.shape[0]

# 3. 动态推算基础特征 X 的维度
w0 = state_dict['shared_layers.0.weight']
total_in_dim = w0.shape[1]
x_main_dim = total_in_dim - c_emb_dim

# 4. 构造极度公平的“克隆人”对照组：
# 基础特征全为 0，唯一的变量就是他们携带的因果 Embedding！
x_dummy = torch.zeros(x_main_dim)
in_nt = torch.cat([x_dummy, emb_never])
in_co = torch.cat([x_dummy, emb_comp])

print("="*60)
print("🚀 [V2 全链路核磁共振]：追踪 NT 和 CO 在深层网络里的存活状态")
print("="*60)

curr_nt = in_nt
curr_co = in_co
layer_idx = 0
layer_count = 1

# 5. 逐层穿透 Shared Bottom (模拟前向传播)
while True:
    w_key = f'shared_layers.{layer_idx}.weight'
    b_key = f'shared_layers.{layer_idx}.bias'
    
    # 如果找不到权重了，说明 Shared Bottom 跑完了
    if w_key not in state_dict:
        break
        
    w = state_dict[w_key]
    b = state_dict[b_key]
    
    # 线性变换 (Linear)
    curr_nt = F.linear(curr_nt, w, b)
    curr_co = F.linear(curr_co, w, b)
    
    # 激活函数 (ReLU)
    curr_nt = F.relu(curr_nt)
    curr_co = F.relu(curr_co)
    
    print(f"\n🧠 [Shared Layer {layer_count}] 经过 ReLU 截断后:")
    print(f"  👉 NT: 幸存(非零)神经元 {(curr_nt > 0).sum().item():>2d} / {len(curr_nt)}, 向量均值: {curr_nt.mean().item():.4f}")
    print(f"  👉 CO: 幸存(非零)神经元 {(curr_co > 0).sum().item():>2d} / {len(curr_co)}, 向量均值: {curr_co.mean().item():.4f}")
    
    # TARNET 的 Sequential 结构通常是 Linear(0) -> ReLU(1) -> Linear(2) -> ReLU(3)...
    # 所以带有 weight 的层索引是 0, 2, 4... (如果有 Dropout 可能会偏移，这里按标准步长推算)
    layer_idx += 1 
    while f'shared_layers.{layer_idx}.weight' not in state_dict and layer_idx < 20:
        layer_idx += 1
    layer_count += 1

# 6. 最终宣判：进入 y0 塔
w_head0 = state_dict['head_0.weight']
b_head0 = state_dict['head_0.bias']

logit_nt = F.linear(curr_nt, w_head0, b_head0).item()
logit_co = F.linear(curr_co, w_head0, b_head0).item()

prob_nt = torch.sigmoid(torch.tensor(logit_nt)).item()
prob_co = torch.sigmoid(torch.tensor(logit_co)).item()

print("\n" + "="*60)
print("🎯 [终极裁决]：走出迷宫，y0 塔 (head_0) 看到的最终结果")
print(f"👦 NT (绝不点击者) -> Logit: {logit_nt:>7.4f} | 最终预测概率 y0: {prob_nt:.6f}")
print(f"😎 CO (敏感点击者) -> Logit: {logit_co:>7.4f} | 最终预测概率 y0: {prob_co:.6f}")
print("="*60)

if prob_co < prob_nt:
    print("\n🚨 破案了！在相同的基础特征下，单单是『CO』这个标签，")
    print("🚨 就足以让他在 y0 塔里被扣掉更多的分，死得比 NT 还惨！")
else:
    print("\n🤔 咦？对于完全空白特征的用户，CO 似乎没那么惨，我们需要看特征交叉的情况。")

🚀 [V2 全链路核磁共振]：追踪 NT 和 CO 在深层网络里的存活状态

🧠 [Shared Layer 1] 经过 ReLU 截断后:
  👉 NT: 幸存(非零)神经元 29 / 64, 向量均值: 0.2809
  👉 CO: 幸存(非零)神经元 40 / 64, 向量均值: 0.2093

🧠 [Shared Layer 2] 经过 ReLU 截断后:
  👉 NT: 幸存(非零)神经元 17 / 32, 向量均值: 0.2321
  👉 CO: 幸存(非零)神经元 17 / 32, 向量均值: 0.1047

🎯 [终极裁决]：走出迷宫，y0 塔 (head_0) 看到的最终结果
👦 NT (绝不点击者) -> Logit: -0.9262 | 最终预测概率 y0: 0.283698
😎 CO (敏感点击者) -> Logit: -0.3017 | 最终预测概率 y0: 0.425149

🤔 咦？对于完全空白特征的用户，CO 似乎没那么惨，我们需要看特征交叉的情况。


In [14]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

# ==========================================
# 1. 加载真实 V2 权重
# ==========================================
ckpt_path = "/NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v2_emb/run_v2_emb/best_model.pth"
print(f"🔍 正在加载 V2 权重: {ckpt_path}")
state_dict = torch.load(ckpt_path, map_location='cpu')
if 'model_state_dict' in state_dict:
    state_dict = state_dict['model_state_dict']

emb_never = state_dict['emb_never']
emb_comp = state_dict['emb_comp']
emb_always = state_dict['emb_always']
c_emb_dim = emb_never.shape[0]

w0 = state_dict['shared_layers.0.weight']
x_main_dim = w0.shape[1] - c_emb_dim

# ==========================================
# 2. 加载真实 Stage 1 预测数据
# ==========================================
csv_path = "/NAS/shith/uplift/results/criteo/train_c/TARNET/c_v1_base/exp_c_explore/test_dist.csv"
print(f"📂 正在加载真实 C 模型预测结果: {csv_path}")
df = pd.read_csv(csv_path)

# 提取真实的概率
c0_prob = torch.tensor(df['y0_prob'].values, dtype=torch.float32)
c1_prob = torch.tensor(df['y1_prob'].values, dtype=torch.float32)

# 完全复刻你源码里的 extract_pi_prior 逻辑
pi_00 = (1.0 - c1_prob).clamp(min=0.0)      # NT
pi_01 = (c1_prob - c0_prob).clamp(min=0.0)  # CO
pi_11 = c0_prob                             # AT

# ==========================================
# 3. 划分出最典型的三类人群 (取概率最高的 Top 2000 人)
# ==========================================
# 找到极其纯粹的群体，便于观察网络行为
top_k = 2000
idx_nt = torch.topk(pi_00, top_k).indices
idx_co = torch.topk(pi_01, top_k).indices
idx_at = torch.topk(pi_11, top_k).indices

print(f"\n👥 成功筛选出三类典型人群 (各 {top_k} 人):")
print(f"  - 典型 NT 的平均 pi_00: {pi_00[idx_nt].mean().item():.4f}")
print(f"  - 典型 CO 的平均 pi_01: {pi_01[idx_co].mean().item():.4f}")
print(f"  - 典型 AT 的平均 pi_11: {pi_11[idx_at].mean().item():.4f}")

# ==========================================
# 4. 构建联合 Embedding (c_feat)
# ==========================================
# 公式: c_feat = pi_00 * emb_never + pi_01 * emb_comp + pi_11 * emb_always
def build_c_feat(indices):
    return (pi_00[indices].unsqueeze(1) * emb_never.unsqueeze(0) + 
            pi_01[indices].unsqueeze(1) * emb_comp.unsqueeze(0) + 
            pi_11[indices].unsqueeze(1) * emb_always.unsqueeze(0))

c_feat_nt = build_c_feat(idx_nt)
c_feat_co = build_c_feat(idx_co)
c_feat_at = build_c_feat(idx_at)

# 绝对公平控制变量：所有人的基础特征 x_main 全部设为 0
x_dummy = torch.zeros(top_k, x_main_dim)

curr_nt = torch.cat([x_dummy, c_feat_nt], dim=1)
curr_co = torch.cat([x_dummy, c_feat_co], dim=1)
curr_at = torch.cat([x_dummy, c_feat_at], dim=1)

# ==========================================
# 5. 逐层穿透 Shared Bottom
# ==========================================
print("\n" + "="*70)
print("🚀 [真实数据 MRI]：追踪 NT, CO, AT 在 V2 网络深处的生与死")
print("="*70)

layer_idx = 0
layer_count = 1

while True:
    w_key = f'shared_layers.{layer_idx}.weight'
    b_key = f'shared_layers.{layer_idx}.bias'
    if w_key not in state_dict: break
        
    w, b = state_dict[w_key], state_dict[b_key]
    
    # Linear + ReLU
    curr_nt = F.relu(F.linear(curr_nt, w, b))
    curr_co = F.relu(F.linear(curr_co, w, b))
    curr_at = F.relu(F.linear(curr_at, w, b))
    
    print(f"\n🧠 [Shared Layer {layer_count}] ReLU 截断后 (均值越小说明被压制越惨):")
    print(f"  👉 典型 NT: 向量均值 = {curr_nt.mean().item():>7.4f} | 零值率: {(curr_nt == 0).float().mean().item()*100:>5.1f}%")
    print(f"  👉 典型 CO: 向量均值 = {curr_co.mean().item():>7.4f} | 零值率: {(curr_co == 0).float().mean().item()*100:>5.1f}%")
    print(f"  👉 典型 AT: 向量均值 = {curr_at.mean().item():>7.4f} | 零值率: {(curr_at == 0).float().mean().item()*100:>5.1f}%")
    
    layer_idx += 1
    while f'shared_layers.{layer_idx}.weight' not in state_dict and layer_idx < 20:
        layer_idx += 1
    layer_count += 1

# ==========================================
# 6. 进入 y0 和 y1 塔进行最终裁决
# ==========================================
w_h0, b_h0 = state_dict['head_0.weight'], state_dict['head_0.bias']
w_h1, b_h1 = state_dict['head_1.weight'], state_dict['head_1.bias']

# 计算 y0 概率
prob_nt_y0 = torch.sigmoid(F.linear(curr_nt, w_h0, b_h0)).mean().item()
prob_co_y0 = torch.sigmoid(F.linear(curr_co, w_h0, b_h0)).mean().item()
prob_at_y0 = torch.sigmoid(F.linear(curr_at, w_h0, b_h0)).mean().item()

# 计算 y1 概率 (作为对比)
prob_nt_y1 = torch.sigmoid(F.linear(curr_nt, w_h1, b_h1)).mean().item()
prob_co_y1 = torch.sigmoid(F.linear(curr_co, w_h1, b_h1)).mean().item()
prob_at_y1 = torch.sigmoid(F.linear(curr_at, w_h1, b_h1)).mean().item()

print("\n" + "="*70)
print("🎯 [终极裁决]：剥除所有原始特征后，仅凭 Causal Embedding，网络定下的『死刑基准线』")
print("="*70)
print(f"{'人群分类':<10} | {'y0 塔预估 (不发券转化率)':<25} | {'y1 塔预估 (发券转化率)':<25}")
print("-" * 70)
print(f"{'👦 典型 NT':<10} | {prob_nt_y0:<25.6f} | {prob_nt_y1:<25.6f}")
print(f"{'😎 典型 CO':<10} | {prob_co_y0:<25.6f} | {prob_co_y1:<25.6f}")
print(f"{'🤑 典型 AT':<10} | {prob_at_y0:<25.6f} | {prob_at_y1:<25.6f}")
print("="*70)

if prob_co_y0 < prob_nt_y0:
    print("\n🚨 铁证如山！你亲眼看到了：在提取的真实概率混合下，")
    print("🚨 CO 的 y0 预估基线被网络压得比 NT 还要低！这就是导致大盘崩盘的元凶！")

🔍 正在加载 V2 权重: /NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v2_emb/run_v2_emb/best_model.pth
📂 正在加载真实 C 模型预测结果: /NAS/shith/uplift/results/criteo/train_c/TARNET/c_v1_base/exp_c_explore/test_dist.csv

👥 成功筛选出三类典型人群 (各 2000 人):
  - 典型 NT 的平均 pi_00: 0.9986
  - 典型 CO 的平均 pi_01: 0.1996
  - 典型 AT 的平均 pi_11: 0.7169

🚀 [真实数据 MRI]：追踪 NT, CO, AT 在 V2 网络深处的生与死

🧠 [Shared Layer 1] ReLU 截断后 (均值越小说明被压制越惨):
  👉 典型 NT: 向量均值 =  0.2805 | 零值率:  54.7%
  👉 典型 CO: 向量均值 =  0.1539 | 零值率:  46.8%
  👉 典型 AT: 向量均值 =  0.1470 | 零值率:  43.4%

🧠 [Shared Layer 2] ReLU 截断后 (均值越小说明被压制越惨):
  👉 典型 NT: 向量均值 =  0.2318 | 零值率:  46.9%
  👉 典型 CO: 向量均值 =  0.1290 | 零值率:  44.9%
  👉 典型 AT: 向量均值 =  0.0963 | 零值率:  44.3%

🎯 [终极裁决]：剥除所有原始特征后，仅凭 Causal Embedding，网络定下的『死刑基准线』
人群分类       | y0 塔预估 (不发券转化率)           | y1 塔预估 (发券转化率)           
----------------------------------------------------------------------
👦 典型 NT    | 0.283903                  | 0.350363                 
😎 典型 CO    | 0.366636                  | 0.429751            

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import roc_auc_score

# ====================== 1. 数据生成（已修复） ======================
N = 20000
np.random.seed(42)
torch.manual_seed(42)

# 3 个 compliance 类型（0:NT, 1:CO, 2:AT）
group_np = np.random.choice([0,1,2], N, p=[0.4, 0.4, 0.2])
group = torch.from_numpy(group_np).long()          # ←←← 关键修复：转为 torch tensor

group_onehot = torch.eye(3)[group]                 # oracle E_causal

# 业务特征 X（CO 稍高意向）
X = torch.randn(N, 8)
X[:, 0] += 0.8 * (group == 1).float() + 0.3 * (group == 2).float()  # intent feat

# 真实潜在结果（极端低转化率）
true_y0 = torch.sigmoid(0.5 * X[:,0] - 1.5 + 0.3*(group==2).float())   # NT/CO base 低，AT 略高
uplift = torch.where(group==1, torch.tensor(0.8), torch.tensor(0.0))
true_y1 = true_y0 + uplift

T = torch.bernoulli(torch.ones(N)*0.5)                  # 随机 treatment
Y_obs = torch.where(T==0, true_y0, true_y1)            # 观测标签

# ====================== 2. 模型定义（保持不变） ======================
class TARNET(nn.Module):
    def __init__(self, use_causal=False):
        super().__init__()
        self.use_causal = use_causal
        in_dim = 8 + (3 if use_causal else 0)
        self.shared = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU()
        )
        self.head0 = nn.Linear(32, 1)
        self.head1 = nn.Linear(32, 1)
        if use_causal:
            self.emb = nn.Parameter(torch.randn(3, 3))   # 可学习 emb_NEVER/COMP/ALWAYS
    
    def forward(self, x, group_emb=None):
        if self.use_causal:
            c_feat = group_emb @ self.emb
            x_aug = torch.cat([x, c_feat], dim=1)
        else:
            x_aug = x
        phi = self.shared(x_aug)
        return torch.sigmoid(self.head0(phi)).squeeze(-1), \
               torch.sigmoid(self.head1(phi)).squeeze(-1)

# ====================== 3. 训练函数（保持不变） ======================
def train_model(model, X, group_onehot, T, Y_obs, epochs=800, lr=0.005):
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    crit = nn.BCELoss()
    for ep in range(epochs):
        y0_pred, y1_pred = model(X, group_onehot)
        loss = crit(y0_pred*(1-T), Y_obs*(1-T)) + crit(y1_pred*T, Y_obs*T)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model

# ====================== 4. 对比实验 ======================
X_t = X.float()
T_t = T.float()
Y_t = Y_obs.float()
g_t = group_onehot.float()

v1 = TARNET(use_causal=False)
v1 = train_model(v1, X_t, g_t, T_t, Y_t)

v2 = TARNET(use_causal=True)
v2 = train_model(v2, X_t, g_t, T_t, Y_t)

# ====================== 5. 评估三大现象 ======================
with torch.no_grad():
    y0_v1, _ = v1(X_t, g_t)
    y0_v2, _ = v2(X_t, g_t)
    
    print("=== 现象1：全局 y0 水位 ===")
    print("V1 global y0 mean:", y0_v1.mean().item())
    print("V2 global y0 mean:", y0_v2.mean().item())
    print("真实 control y0 mean:", true_y0[T==0].mean().item())
    
    print("\n=== 现象2：CO vs NT y0（V2） ===")
    mask_nt = (group == 0)
    mask_co = (group == 1)
    print("V2 CO y0 mean:", y0_v2[mask_co].mean().item())
    print("V2 NT y0 mean:", y0_v2[mask_nt].mean().item())
    
    print("\n=== 现象3：同 CO 群体 V1 vs V2 ===")
    print("V1 CO y0 mean:", y0_v1[mask_co].mean().item())
    print("V2 CO y0 mean:", y0_v2[mask_co].mean().item())
    
    print("\n=== 额外验证：X=0 时是否 CO > NT ===")
    X_zero = torch.zeros_like(X_t)
    y0_v2_zero, _ = v2(X_zero, g_t)
    print("X=0 时 V2 CO y0 > NT？", 
          y0_v2_zero[mask_co].mean().item() > y0_v2_zero[mask_nt].mean().item())

RuntimeError: all elements of target should be between 0 and 1

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim

# ================= 1. 构造极度不平衡的模拟数据 =================
N = 10000
# 假设 80% 是 NT，20% 是 CO
is_co = (torch.rand(N) < 0.2).float() 
is_nt = 1.0 - is_co

# CO 人群倾向于有高意向特征 X=1 (80%概率)，NT 倾向于 X=0 (20%概率)
x_main = torch.where(is_co == 1, (torch.rand(N) < 0.8).float(), (torch.rand(N) < 0.2).float()).unsqueeze(1)
c_feat = torch.cat([is_nt.unsqueeze(1), is_co.unsqueeze(1)], dim=1) # [N, 2] One-hot for {NT, CO}

# 构造 T 和 Y
T = (torch.rand(N) < 0.5).float()
Y = torch.zeros(N)

# 真实转化率极低设定:
# 对于 T=0 (对照组): 无论 NT 还是 CO，基础转化率都极低 (0.01)
mask_t0 = (T == 0)
Y[mask_t0] = (torch.rand(mask_t0.sum()) < 0.01).float()

# 对于 T=1 (实验组): CO 转化率提升至 0.05, NT 依然是 0.01
mask_t1_co = (T == 1) & (is_co == 1)
mask_t1_nt = (T == 1) & (is_nt == 1)
Y[mask_t1_co] = (torch.rand(mask_t1_co.sum()) < 0.05).float()
Y[mask_t1_nt] = (torch.rand(mask_t1_nt.sum()) < 0.01).float()

Y = Y.unsqueeze(1)
T = T.unsqueeze(1)

# ================= 2. 定义极简 TARNET =================
class ToyTARNET(nn.Module):
    def __init__(self, use_c_feat=False):
        super().__init__()
        self.use_c_feat = use_c_feat
        in_dim = 3 if use_c_feat else 1 # X(1) + C(2) = 3
        # 模拟 Shared Bottom 产生特征交叉的空间
        self.shared = nn.Sequential(nn.Linear(in_dim, 8), nn.ReLU(), nn.Linear(8, 4), nn.ReLU())
        self.head_0 = nn.Linear(4, 1)
        self.head_1 = nn.Linear(4, 1)
        
    def forward(self, x, c):
        inp = torch.cat([x, c], dim=1) if self.use_c_feat else x
        shared_rep = self.shared(inp)
        y0 = torch.sigmoid(self.head_0(shared_rep))
        y1 = torch.sigmoid(self.head_1(shared_rep))
        return y0, y1

# ================= 3. 训练与验证 =================
def train_model(model_name, use_c_feat):
    model = ToyTARNET(use_c_feat)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.BCELoss()
    
    for epoch in range(1000):
        optimizer.zero_grad()
        y0, y1 = model(x_main, c_feat)
        # TARNET Loss
        loss = (1 - T) * criterion(y0, Y) + T * criterion(y1, Y)
        loss.mean().backward()
        optimizer.step()
        
    # 测试 CO 人群 (高特征 X=1) 在该模型下的 y0 预估
    test_x = torch.tensor([[1.0]])
    test_c_co = torch.tensor([[0.0, 1.0]]) # NT=0, CO=1
    test_c_nt = torch.tensor([[1.0, 0.0]]) # NT=1, CO=0
    
    y0_co, _ = model(test_x, test_c_co)
    y0_nt, _ = model(test_x, test_c_nt)
    
    print(f"[{model_name}] 对于同样的高意向特征 X=1:")
    print(f"   -> 识别为 CO 时, y0 预估 = {y0_co.item():.4f}")
    if use_c_feat:
        print(f"   -> 识别为 NT 时, y0 预估 = {y0_nt.item():.4f}")
        # 测试 X=0 时的情况 (验证排查现状)
        y0_co_x0, _ = model(torch.tensor([[0.0]]), test_c_co)
        y0_nt_x0, _ = model(torch.tensor([[0.0]]), test_c_nt)
        print(f"   [Debug] 当强行置空 X=0 时: CO_y0={y0_co_x0.item():.4f}, NT_y0={y0_nt_x0.item():.4f}")

train_model("V1 (仅用 X)", use_c_feat=False)
print("-" * 40)
train_model("V2 (使用 X + 因果 Embedding)", use_c_feat=True)

[V1 (仅用 X)] 对于同样的高意向特征 X=1:
   -> 识别为 CO 时, y0 预估 = 0.0226
----------------------------------------
[V2 (使用 X + 因果 Embedding)] 对于同样的高意向特征 X=1:
   -> 识别为 CO 时, y0 预估 = 0.0336
   -> 识别为 NT 时, y0 预估 = 0.0119
   [Debug] 当强行置空 X=0 时: CO_y0=0.0270, NT_y0=0.0095


In [20]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

# ==========================================
# 1. 加载真实 V2 权重
# ==========================================
ckpt_path = "/NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v2_emb/run_v2_emb/best_model.pth"
print(f"🔍 正在加载 V2 权重: {ckpt_path}")
state_dict = torch.load(ckpt_path, map_location='cpu')
if 'model_state_dict' in state_dict:
    state_dict = state_dict['model_state_dict']

emb_never = state_dict['emb_never']
emb_comp = state_dict['emb_comp']
emb_always = state_dict['emb_always']
c_emb_dim = emb_never.shape[0]

w0 = state_dict['shared_layers.0.weight']
x_main_dim = w0.shape[1] - c_emb_dim

# ==========================================
# 2. 加载真实 Stage 1 预测数据 (取全局大盘，不只取 Top)
# ==========================================
csv_path = "/NAS/shith/uplift/results/criteo/train_c/TARNET/c_v1_base/exp_c_explore/test_dist.csv"
print(f"📂 正在加载真实 C 模型预测结果: {csv_path}")
df = pd.read_csv(csv_path)

# 取全局样本（为了内存考虑可以采样 50000 条，验证全局均值足够了）
sample_size = min(50000, len(df))
df_sample = df.sample(n=sample_size, random_state=42)

c0_prob = torch.tensor(df_sample['y0_prob'].values, dtype=torch.float32)
c1_prob = torch.tensor(df_sample['y1_prob'].values, dtype=torch.float32)

pi_00 = (1.0 - c1_prob).clamp(min=0.0)      
pi_01 = (c1_prob - c0_prob).clamp(min=0.0)  
pi_11 = c0_prob                             

# 构建全局真实的联合 Embedding
c_feat_real = (pi_00.unsqueeze(1) * emb_never.unsqueeze(0) + 
               pi_01.unsqueeze(1) * emb_comp.unsqueeze(0) + 
               pi_11.unsqueeze(1) * emb_always.unsqueeze(0))

# 构建消融 Embedding (全 0)
c_feat_zero = torch.zeros_like(c_feat_real)

# 模拟真实的业务特征 X (假设经过归一化，均值0，方差1)
# ⚠️ 强烈建议：如果你有真实的 X 矩阵，替换掉这行代码！效果最准。
x_main = torch.randn(sample_size, x_main_dim)

print(f"\n📊 大盘样本准备完毕，样本量: {sample_size}")
print(f"  - 因果 Embedding (c_feat) 的全局绝对均值 (Norm): {c_feat_real.abs().mean().item():.6f}")

# 拼接：一个带 Embedding，一个切断 Embedding
input_with_emb = torch.cat([x_main, c_feat_real], dim=1)
input_without_emb = torch.cat([x_main, c_feat_zero], dim=1)

# ==========================================
# 3. 逐层穿透 Shared Bottom：追踪“水位漂移”
# ==========================================
print("\n" + "="*70)
print("🌊 [水位监测仪]：对比加入 Emb 前后，Shared Bottom 的激活值偏移")
print("="*70)

curr_with = input_with_emb
curr_without = input_without_emb
layer_idx = 0
layer_count = 1

while True:
    w_key = f'shared_layers.{layer_idx}.weight'
    b_key = f'shared_layers.{layer_idx}.bias'
    if w_key not in state_dict: break
        
    w, b = state_dict[w_key], state_dict[b_key]
    
    # Linear + ReLU
    curr_with = F.relu(F.linear(curr_with, w, b))
    curr_without = F.relu(F.linear(curr_without, w, b))
    
    print(f"\n🧠 [Shared Layer {layer_count}] ReLU 激活后状态:")
    print(f"  👉 【含 Emb (V2 真实)】: 均值 = {curr_with.mean().item():>8.4f} | 方差 = {curr_with.std().item():>8.4f} | 零值率: {(curr_with == 0).float().mean().item()*100:>5.1f}%")
    print(f"  👉 【无 Emb (模拟 V1)】: 均值 = {curr_without.mean().item():>8.4f} | 方差 = {curr_without.std().item():>8.4f} | 零值率: {(curr_without == 0).float().mean().item()*100:>5.1f}%")
    
    # 诊断：均值差异是否过大？
    diff = curr_with.mean().item() - curr_without.mean().item()
    if diff > 0.5:
        print(f"  🚨 警告：本层因果 Emb 带来了强烈的正向激活偏移 (+{diff:.4f})！")
    
    layer_idx += 1
    while f'shared_layers.{layer_idx}.weight' not in state_dict and layer_idx < 20:
        layer_idx += 1
    layer_count += 1

# ==========================================
# 4. 进入 y0 和 y1 塔：揭晓最终均值
# ==========================================
w_h0, b_h0 = state_dict['head_0.weight'], state_dict['head_0.bias']
w_h1, b_h1 = state_dict['head_1.weight'], state_dict['head_1.bias']

# 最终预估
prob_y0_with_emb = torch.sigmoid(F.linear(curr_with, w_h0, b_h0))
prob_y0_without_emb = torch.sigmoid(F.linear(curr_without, w_h0, b_h0))

print("\n" + "="*70)
print("🎯 [全局均值裁决]：因果 Embedding 是否是拉高大盘的罪魁祸首？")
print("="*70)
print(f" 📈 真实 CVR (参考大盘)         : ~ 0.002000")
print(f" 🔴 【V2 真实】全局 y0 预估均值: {prob_y0_with_emb.mean().item():.6f}")
print(f" 🟢 【切断 Emb】全局 y0 预估均值: {prob_y0_without_emb.mean().item():.6f}")
print("-" * 70)

# 诊断结论
if prob_y0_with_emb.mean().item() > 0.005 and prob_y0_without_emb.mean().item() < 0.005:
    print("🚨 铁证如山：切断因果 Embedding 后，y0 预估水位瞬间恢复正常！")
    print("👉 结论 1：量纲污染或多任务梯度牵引导致 Embedding 整体携带了巨大的正偏置，直接把 y0 的输出强行抬高了！")
elif prob_y0_with_emb.mean().item() > 0.005 and prob_y0_without_emb.mean().item() > 0.005:
    print("🤔 有点意思：切断 Embedding 后，y0 依然偏高。")
    print("👉 结论 2：说明 Shared Bottom 已经被 V2 的训练过程整体破坏了（参数权重 w 已经被带偏），单纯推理时置 0 救不回来。")

print("\n" + "="*70)
print("🔍 [网络极性探针]：查看 Head 的 Bias 是否在死命抵抗")
print("="*70)
print(f"  - y0 Head Bias: {b_h0.item():.4f}")
print(f"  - y1 Head Bias: {b_h1.item():.4f}")
if b_h0.item() < -5.0:
    print("  🚨 y0 Bias 极度负向！模型在拼命尝试把上漂的 Logit 压下来拟合 0.002，但压不住！")

🔍 正在加载 V2 权重: /NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v2_emb/run_v2_emb/best_model.pth
📂 正在加载真实 C 模型预测结果: /NAS/shith/uplift/results/criteo/train_c/TARNET/c_v1_base/exp_c_explore/test_dist.csv

📊 大盘样本准备完毕，样本量: 50000
  - 因果 Embedding (c_feat) 的全局绝对均值 (Norm): 1.044185

🌊 [水位监测仪]：对比加入 Emb 前后，Shared Bottom 的激活值偏移

🧠 [Shared Layer 1] ReLU 激活后状态:
  👉 【含 Emb (V2 真实)】: 均值 =   0.2996 | 方差 =   0.4632 | 零值率:  53.2%
  👉 【无 Emb (模拟 V1)】: 均值 =   0.1502 | 方差 =   0.2200 | 零值率:  48.7%

🧠 [Shared Layer 2] ReLU 激活后状态:
  👉 【含 Emb (V2 真实)】: 均值 =   0.2421 | 方差 =   0.3253 | 零值率:  44.1%
  👉 【无 Emb (模拟 V1)】: 均值 =   0.0911 | 方差 =   0.1396 | 零值率:  48.6%

🎯 [全局均值裁决]：因果 Embedding 是否是拉高大盘的罪魁祸首？
 📈 真实 CVR (参考大盘)         : ~ 0.002000
 🔴 【V2 真实】全局 y0 预估均值: 0.285275
 🟢 【切断 Emb】全局 y0 预估均值: 0.408247
----------------------------------------------------------------------
🤔 有点意思：切断 Embedding 后，y0 依然偏高。
👉 结论 2：说明 Shared Bottom 已经被 V2 的训练过程整体破坏了（参数权重 w 已经被带偏），单纯推理时置 0 救不回来。

🔍 [网络极性探针]：查看 Head 的 Bias 是否在死命抵抗
  - y0 Hea

In [21]:
import torch
import numpy as np

def load_ckpt(path):
    print(f"📥 加载权重: {path}")
    state_dict = torch.load(path, map_location='cpu')
    return state_dict.get('model_state_dict', state_dict)

v1_path = "/NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v1_base/run_v1_base/best_model.pth"
v2_path = "/NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v2_emb/run_v2_emb/best_model.pth"

sd_v1 = load_ckpt(v1_path)
sd_v2 = load_ckpt(v2_path)

print("\n" + "="*70)
print("⚖️ [第一回合]：顶层 Bias 对比 (大盘基线锚点)")
print("="*70)
b0_v1 = sd_v1['head_0.bias'].item()
b1_v1 = sd_v1['head_1.bias'].item()
b0_v2 = sd_v2['head_0.bias'].item()
b1_v2 = sd_v2['head_1.bias'].item()

# 理论上的完美 Bias 应该在 -6.21 左右 (对应 CVR 0.002)
print(f" 🟢 V1 (基线模型): y0_bias = {b0_v1:>8.4f} | y1_bias = {b1_v1:>8.4f}")
print(f" 🔴 V2 (引入 Emb): y0_bias = {b0_v2:>8.4f} | y1_bias = {b1_v2:>8.4f}")

if abs(b0_v1 - b0_v2) > 1.0:
    print(" 👉 诊断: V1 和 V2 在基线学习上出现了巨大分歧！看看是谁更接近 -6.0。")

print("\n" + "="*70)
print("💪 [第二回合]：V2 内部 Embedding 的绝对力量 (L2 Norm)")
print("="*70)
emb_never = sd_v2['emb_never']
emb_comp = sd_v2['emb_comp']
emb_always = sd_v2['emb_always']

print(f"  - emb_never (NT)  L2范数: {torch.norm(emb_never).item():.4f} | 均值: {emb_never.mean().item():.4f}")
print(f"  - emb_comp  (CO)  L2范数: {torch.norm(emb_comp).item():.4f} | 均值: {emb_comp.mean().item():.4f}")
print(f"  - emb_always(AT)  L2范数: {torch.norm(emb_always).item():.4f} | 均值: {emb_always.mean().item():.4f}")

print("\n" + "="*70)
print("🕵️ [第三回合]：Shared Bottom 第一层权重注意力争夺战 (Column Norm)")
print("="*70)
w0_v1 = sd_v1['shared_layers.0.weight'] # shape: [out_features, in_features_x]
w0_v2 = sd_v2['shared_layers.0.weight'] # shape: [out_features, in_features_x + emb_dim]

c_emb_dim = emb_never.shape[0]
x_main_dim = w0_v2.shape[1] - c_emb_dim

# 计算列范数 (每个输入特征对应的权重向量的 L2 长度，代表该特征被网络放大的倍数)
# V1 只有 X
norm_v1_x = torch.norm(w0_v1, dim=0).mean().item()

# V2 有 X 也有 Causal Emb
norm_v2_x = torch.norm(w0_v2[:, :x_main_dim], dim=0).mean().item()
norm_v2_c = torch.norm(w0_v2[:, x_main_dim:], dim=0).mean().item()

print(f" 🟢 V1 网络对业务特征 X 的平均关注度 (Norm): {norm_v1_x:.4f}")
print(f" 🔴 V2 网络对业务特征 X 的平均关注度 (Norm): {norm_v2_x:.4f}")
print(f" 🔴 V2 网络对因果 Emb C 的平均关注度 (Norm): {norm_v2_c:.4f}")
print("-" * 70)

if norm_v2_c > norm_v2_x:
    ratio = norm_v2_c / norm_v2_x
    print(f" 🚨 铁证如山：在 V2 的底层，网络对因果 Embedding 的依赖程度是对普通特征 X 的 {ratio:.1f} 倍！")
    print("    它彻底喧宾夺主，变成了主导 Shared Bottom 激活值的核心特征！")
elif norm_v2_x < norm_v1_x * 0.5:
    print(" 🚨 特征遮蔽：引入 Emb 后，V2 对原有业务特征 X 的权重被大幅度削弱了 (Feature Overshadowing)。")

📥 加载权重: /NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v1_base/run_v1_base/best_model.pth
📥 加载权重: /NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v2_emb/run_v2_emb/best_model.pth

⚖️ [第一回合]：顶层 Bias 对比 (大盘基线锚点)
 🟢 V1 (基线模型): y0_bias =  -0.0238 | y1_bias =   0.1237
 🔴 V2 (引入 Emb): y0_bias =  -0.1161 | y1_bias =   0.1282

💪 [第二回合]：V2 内部 Embedding 的绝对力量 (L2 Norm)
  - emb_never (NT)  L2范数: 5.6467 | 均值: -0.1941
  - emb_comp  (CO)  L2范数: 3.8449 | 均值: 0.2967
  - emb_always(AT)  L2范数: 3.6509 | 均值: 0.3217

🕵️ [第三回合]：Shared Bottom 第一层权重注意力争夺战 (Column Norm)
 🟢 V1 网络对业务特征 X 的平均关注度 (Norm): 1.8449
 🔴 V2 网络对业务特征 X 的平均关注度 (Norm): 0.8304
 🔴 V2 网络对因果 Emb C 的平均关注度 (Norm): 0.8009
----------------------------------------------------------------------
 🚨 特征遮蔽：引入 Emb 后，V2 对原有业务特征 X 的权重被大幅度削弱了 (Feature Overshadowing)。


In [23]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd

# ==========================================
# 1. 绝对准确的路径配置 (基于你提供的真实环境)
# ==========================================
v1_path = "/NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v1_base/run_v1_base/best_model.pth"
v2_path = "/NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v2_emb/run_v2_emb/best_model.pth"
parquet_path = "/NAS/shith/datasets2024/uplift/criteo/test.parquet"
c_pred_path = "/NAS/shith/uplift/results/criteo/train_c/TARNET/c_v1_base/exp_c_explore/test_dist.csv"

print("🚀 开始执行：V2 模型特征致盲与坍塌终极测试\n")

# ==========================================
# 2. 加载真实数据 (取前 20000 条进行严格对齐测试)
# ==========================================
print("📂 正在加载真实特征与 C 模型预测分布...")
limit = 20000

# 读取连续业务特征 X (f0 到 f11)
df_x = pd.read_parquet(parquet_path).head(limit)
x_cols = [f"f{i}" for i in range(12)]
x_real = torch.tensor(df_x[x_cols].values, dtype=torch.float32)

# 读取 C 模型的预测概率
df_c = pd.read_csv(c_pred_path).head(limit)
c0_prob = torch.tensor(df_c['y0_prob'].values, dtype=torch.float32)
c1_prob = torch.tensor(df_c['y1_prob'].values, dtype=torch.float32)

# 计算先验
pi_00 = (1.0 - c1_prob).clamp(min=0.0)
pi_01 = (c1_prob - c0_prob).clamp(min=0.0)
pi_11 = c0_prob

# ==========================================
# 3. 加载模型权重与 Embedding
# ==========================================
print("📥 正在加载 V1 和 V2 权重...")
sd_v1 = torch.load(v1_path, map_location='cpu')
sd_v1 = sd_v1.get('model_state_dict', sd_v1)

sd_v2 = torch.load(v2_path, map_location='cpu')
sd_v2 = sd_v2.get('model_state_dict', sd_v2)

# 提取 V2 的 Embedding
emb_never = sd_v2['emb_never']
emb_comp = sd_v2['emb_comp']
emb_always = sd_v2['emb_always']

# 构造真实的 c_feat (仅供 V2 使用)
c_feat_real = (pi_00.unsqueeze(1) * emb_never.unsqueeze(0) +
               pi_01.unsqueeze(1) * emb_comp.unsqueeze(0) +
               pi_11.unsqueeze(1) * emb_always.unsqueeze(0))

# ==========================================
# 4. 定义极简的前向传播 (剥离任何复杂的 Class 依赖)
# ==========================================
def manual_forward(x_input, c_feat_input, state_dict):
    """手动穿透网络，绝不会被其他代码逻辑污染"""
    # 如果有因果 Emb，就拼接 (V2 逻辑)；没有就只用 x (V1 逻辑)
    curr = torch.cat([x_input, c_feat_input], dim=1) if c_feat_input is not None else x_input
    
    # 动态穿透 Shared Bottom
    for i in range(20):
        w_key = f'shared_layers.{i}.weight'
        b_key = f'shared_layers.{i}.bias'
        if w_key in state_dict:
            curr = F.relu(F.linear(curr, state_dict[w_key], state_dict[b_key]))
            
    # 穿透 y0 Head
    y0 = torch.sigmoid(F.linear(curr, state_dict['head_0.weight'], state_dict['head_0.bias']))
    return y0.squeeze()

# ==========================================
# 5. 核心实验 A：方差坍塌测试 (Variance Collapse)
# ==========================================
print("\n" + "="*70)
print("🔬 实验 A：观察 V1 和 V2 在正常状态下的预估区分度")
print("="*70)

y0_v1_norm = manual_forward(x_real, None, sd_v1)
y0_v2_norm = manual_forward(x_real, c_feat_real, sd_v2)

var_v1 = y0_v1_norm.var().item()
var_v2 = y0_v2_norm.var().item()

print(f" 🟢 V1 正常预估的方差 (区分度): {var_v1:.8f}")
print(f" 🔴 V2 正常预估的方差 (区分度): {var_v2:.8f}")

if var_v2 < var_v1 / 5:
    print(" 🚨 结论 A：V2 的方差极其微小！它彻底丧失了对用户的个性化排序能力，预估值坍塌成了一条水平的死线。")

# ==========================================
# 6. 核心实验 B：业务特征致盲测试 (Feature Blindness)
# ==========================================
print("\n" + "="*70)
print("🙈 实验 B：把业务特征 X 彻底打乱，保留 Embedding 不变，看模型瞎不瞎？")
print("="*70)

# 创造一个被彻底打乱的 X，但用户对应的 Causal Embedding 保持原样！
idx_shuffled = torch.randperm(limit)
x_shuffled = x_real[idx_shuffled]

y0_v1_shuf = manual_forward(x_shuffled, None, sd_v1)
y0_v2_shuf = manual_forward(x_shuffled, c_feat_real, sd_v2)

# 计算打乱 X 前后，每个用户预估分数的平均绝对误差 (MAE)
diff_v1 = torch.abs(y0_v1_norm - y0_v1_shuf).mean().item()
diff_v2 = torch.abs(y0_v2_norm - y0_v2_shuf).mean().item()

print(f" 🟢 打乱 X 后，V1 预估值的平均变化幅度: {diff_v1:.6f}")
print(f" 🔴 打乱 X 后，V2 预估值的平均变化幅度: {diff_v2:.6f}")
print("-" * 70)

if diff_v2 < diff_v1 / 5:
    print("🚨 铁证如山：")
    print(" 1. V1 在特征打乱后，预估分发生了剧烈变化，说明它在认真读取 X。")
    print(" 2. V2 在特征打乱后，预估分竟然纹丝不动！")
    print(" 3. 终极真相：V2 根本就不看业务特征 X 了！底层网络被 Embedding 完全霸占。")
    print(" 4. 解释：既然它不看 X，它就根本不知道你传进来的 CO 人群原本的转化率有多高。它只是为了强行凑出 uplift 的正数差值，闭着眼睛把所有带 CO 标签的人的 y0 往下踩了一点点而已。")

🚀 开始执行：V2 模型特征致盲与坍塌终极测试

📂 正在加载真实特征与 C 模型预测分布...
📥 正在加载 V1 和 V2 权重...

🔬 实验 A：观察 V1 和 V2 在正常状态下的预估区分度
 🟢 V1 正常预估的方差 (区分度): 0.00006684
 🔴 V2 正常预估的方差 (区分度): 0.00000261
 🚨 结论 A：V2 的方差极其微小！它彻底丧失了对用户的个性化排序能力，预估值坍塌成了一条水平的死线。

🙈 实验 B：把业务特征 X 彻底打乱，保留 Embedding 不变，看模型瞎不瞎？
 🟢 打乱 X 后，V1 预估值的平均变化幅度: 0.003707
 🔴 打乱 X 后，V2 预估值的平均变化幅度: 0.001826
----------------------------------------------------------------------


In [24]:
def manual_forward_y1(x_input, c_feat_input, state_dict):
    curr = torch.cat([x_input, c_feat_input], dim=1) if c_feat_input is not None else x_input
    for i in range(20):
        w_key = f'shared_layers.{i}.weight'
        b_key = f'shared_layers.{i}.bias'
        if w_key in state_dict:
            curr = F.relu(F.linear(curr, state_dict[w_key], state_dict[b_key]))
    # 切换到 y1 head
    y1 = torch.sigmoid(F.linear(curr, state_dict['head_1.weight'], state_dict['head_1.bias']))
    return y1.squeeze()

y0_v2 = manual_forward(x_real, c_feat_real, sd_v2)
y1_v2 = manual_forward_y1(x_real, c_feat_real, sd_v2)

print(f"🔴 V2 y0 均值: {y0_v2.mean().item():.6f} | 方差: {y0_v2.var().item():.8f}")
print(f"🔵 V2 y1 均值: {y1_v2.mean().item():.6f} | 方差: {y1_v2.var().item():.8f}")
print(f"📊 V2 平均 Uplift (y1-y0): {(y1_v2 - y0_v2).mean().item():.6f}")

🔴 V2 y0 均值: 0.006628 | 方差: 0.00000261
🔵 V2 y1 均值: 0.002560 | 方差: 0.00000046
📊 V2 平均 Uplift (y1-y0): -0.004069


In [26]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd

# ==========================================
# 1. 加载模型与真实数据 (复用你之前的路径)
# ==========================================
v1_path = "/NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v1_base/run_v1_base/best_model.pth"
v2_path = "/NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v2_emb/run_v2_emb/best_model.pth"
parquet_path = "/NAS/shith/datasets2024/uplift/criteo/test.parquet"
c_pred_path = "/NAS/shith/uplift/results/criteo/train_c/TARNET/c_v1_base/exp_c_explore/test_dist.csv"

sd_v1 = torch.load(v1_path, map_location='cpu')
sd_v1 = sd_v1.get('model_state_dict', sd_v1)
sd_v2 = torch.load(v2_path, map_location='cpu')
sd_v2 = sd_v2.get('model_state_dict', sd_v2)

# 读取小批量真实数据用于解剖
limit = 5000
df_x = pd.read_parquet(parquet_path).head(limit)
x_real = torch.tensor(df_x[[f"f{i}" for i in range(12)]].values, dtype=torch.float32)

df_c = pd.read_csv(c_pred_path).head(limit)
c0_prob, c1_prob = torch.tensor(df_c['y0_prob'].values, dtype=torch.float32), torch.tensor(df_c['y1_prob'].values, dtype=torch.float32)
pi_00 = (1.0 - c1_prob).clamp(min=0.0)
pi_01 = (c1_prob - c0_prob).clamp(min=0.0)
pi_11 = c0_prob

emb_never, emb_comp, emb_always = sd_v2['emb_never'], sd_v2['emb_comp'], sd_v2['emb_always']
c_feat_real = (pi_00.unsqueeze(1) * emb_never.unsqueeze(0) + pi_01.unsqueeze(1) * emb_comp.unsqueeze(0) + pi_11.unsqueeze(1) * emb_always.unsqueeze(0))

def get_shared_rep(x_input, c_feat_input, state_dict):
    """只提取 Shared Bottom 最后一层的激活状态"""
    curr = torch.cat([x_input, c_feat_input], dim=1) if c_feat_input is not None else x_input
    for i in range(20):
        w_key, b_key = f'shared_layers.{i}.weight', f'shared_layers.{i}.bias'
        if w_key in state_dict:
            curr = F.relu(F.linear(curr, state_dict[w_key], state_dict[b_key]))
    return curr

# ==========================================
# 🔍 验证假说 A：谁在承担“下压 Logit”的重任？
# ==========================================
print("\n" + "="*70)
print("🔬 假说 A 验证：Logit 下压力量解剖 (Bias vs 权重)")
print("="*70)

rep_v1 = get_shared_rep(x_real, None, sd_v1)
rep_v2 = get_shared_rep(x_real, c_feat_real, sd_v2)

# V1 的 Logit 组成
bias_v1 = sd_v1['head_0.bias'].item()
weight_push_v1 = F.linear(rep_v1, sd_v1['head_0.weight']).mean().item()

# V2 的 Logit 组成
bias_v2 = sd_v2['head_0.bias'].item()
weight_push_v2 = F.linear(rep_v2, sd_v2['head_0.weight']).mean().item()

print(f"🎯 目标：达到大盘 0.002 水位，需要最终 Logit 达到约 -6.21")
print(f" 🟢 [V1] 最终 Logit均值: {bias_v1 + weight_push_v1:.4f} = (Bias: {bias_v1:.4f}) + (特征下压力: {weight_push_v1:.4f})")
print(f" 🔴 [V2] 最终 Logit均值: {bias_v2 + weight_push_v2:.4f} = (Bias: {bias_v2:.4f}) + (特征下压力: {weight_push_v2:.4f})")

# 统计最后一层权重的负极性
neg_w_v1 = (sd_v1['head_0.weight'] < 0).float().mean().item()
neg_w_v2 = (sd_v2['head_0.weight'] < 0).float().mean().item()
print(f"\n🧠 y0 Head 权重的『灭火器』比例 (负权重占比):")
print(f" 🟢 V1 负权重占比: {neg_w_v1*100:.1f}%")
print(f" 🔴 V2 负权重占比: {neg_w_v2*100:.1f}%")

if neg_w_v2 > 0.6:
    print(" 🚨 结论 A 实锤：V2 没有通过降低 Bias (-0.11) 来贴合大盘，而是让 y0 Head 长出了大量负权重，强行把底层的热量 (0.24均值) 踩了下去！这耗费了模型巨大的表现力。")

# ==========================================
# 📉 验证假说 B：BCE 梯度消失的死亡地带
# ==========================================
print("\n" + "="*70)
print("📉 假说 B 验证：极度不平衡下的 BCE 梯度衰减计算")
print("="*70)
# 数学原理：BCE Loss 对 Logit (z) 的偏导数梯度就是 (Pred_Prob - True_Label)
# 在大盘负样本 (y=0) 上，梯度大小严格等于预测概率本身！

probs = [0.500, 0.050, 0.006, 0.002]
print("当真实标签 y=0 (占大盘 99.8%) 时，模型收到用于修正参数的『惩罚梯度』大小：")
for p in probs:
    grad_magnitude = p - 0.0 # 绝对值
    print(f"  🌊 当预测概率在 {p:>5.3f} 时，回传梯度大小 = {grad_magnitude:.5f}")

ratio = 0.500 / 0.006
print(f"\n🚨 结论 B 实锤：")
print(f" 当模型靠着特征强行把预测值从 0.5 压到 0.006 之后，它受到的梯度推力骤降了 {ratio:.1f} 倍！")
print(" 在 0.006 这个位置，回传的梯度极其微弱 (仅 0.006)。此时如果还要面对 Adam 优化器的权重衰减 (Weight Decay)、或者多任务 y1 塔传来的结构性阻力，模型就会直接【躺平】，陷入局部鞍点，再也没有力气推到 0.002 了。")


🔬 假说 A 验证：Logit 下压力量解剖 (Bias vs 权重)
🎯 目标：达到大盘 0.002 水位，需要最终 Logit 达到约 -6.21
 🟢 [V1] 最终 Logit均值: -7.4362 = (Bias: -0.0238) + (特征下压力: -7.4124)
 🔴 [V2] 最终 Logit均值: -5.0555 = (Bias: -0.1161) + (特征下压力: -4.9394)

🧠 y0 Head 权重的『灭火器』比例 (负权重占比):
 🟢 V1 负权重占比: 62.5%
 🔴 V2 负权重占比: 53.1%

📉 假说 B 验证：极度不平衡下的 BCE 梯度衰减计算
当真实标签 y=0 (占大盘 99.8%) 时，模型收到用于修正参数的『惩罚梯度』大小：
  🌊 当预测概率在 0.500 时，回传梯度大小 = 0.50000
  🌊 当预测概率在 0.050 时，回传梯度大小 = 0.05000
  🌊 当预测概率在 0.006 时，回传梯度大小 = 0.00600
  🌊 当预测概率在 0.002 时，回传梯度大小 = 0.00200

🚨 结论 B 实锤：
 当模型靠着特征强行把预测值从 0.5 压到 0.006 之后，它受到的梯度推力骤降了 83.3 倍！
 在 0.006 这个位置，回传的梯度极其微弱 (仅 0.006)。此时如果还要面对 Adam 优化器的权重衰减 (Weight Decay)、或者多任务 y1 塔传来的结构性阻力，模型就会直接【躺平】，陷入局部鞍点，再也没有力气推到 0.002 了。


In [39]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import json

# ==========================================
# 0. 导入你的模型类 (请确保路径或包名正确)
# ==========================================
# from your_model_file import TARNET_Baseline, TARNET_Proposed

# ==========================================
# 1. 路径与全局配置
# ==========================================
V1_PATH = "/NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v1_base/run_v1_base/best_model.pth"
V2_PATH = "/NAS/shith/uplift/ckpts/criteo/train_y/TARNET/y_v2_emb/run_v2_emb/best_model.pth"
PARQUET_PATH = "/NAS/shith/datasets2024/uplift/criteo/test.parquet"

# C 模型的路径与配置
C_MODEL_PATH = "/NAS/shith/uplift/ckpts/criteo/train_c/TARNET/c_v1_base/exp_c_explore/best_model.pth"
C_CONFIG_PATH = "/NAS/shith/uplift/ckpts/criteo/train_c/TARNET/c_v1_base/exp_c_explore/best_config.json"
C_PRED_PATH = "/NAS/shith/uplift/results/criteo/train_c/TARNET/c_v1_base/exp_c_explore/test_dist.csv"

TRUE_AVG_CTR = 0.002
TARGET_LOGIT = np.log(TRUE_AVG_CTR / (1 - TRUE_AVG_CTR))
DEVICE = torch.device("cpu")

# ==========================================
# 2. 诊断引擎 (直接接收实例化好的模型)
# ==========================================
def get_grad_stats_robust(model, ckpt_path, x_cont, pi_00, pi_01):
    # 加载当前塔 (V1 或 V2) 的权重
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt.get('model_state_dict', ckpt), strict=False)
    model.train() # 开启梯度追踪
    model.zero_grad()
    
    # 前向传播 (TARNET_Baseline 和 TARNET_Proposed 都返回 3 个值)
    y0, y1, _ = model(x_cont, x_cat=None)
        
    # --- 模拟底层标签冲突 ---
    # y0 塔：统一压制水位 (目标为 0)
    loss_y0 = F.binary_cross_entropy_with_logits(y0, torch.zeros_like(y0))
    # y1 塔：CO 样本有正向转化倾向 (目标为 pi_01)
    loss_y1 = F.binary_cross_entropy_with_logits(y1, pi_01)

    # 提取 Shared Bottom 的第一层权重 (即直接与输入相连的那一层)
    first_layer_w = next(model.shared_layers.parameters())
    
    # 分别计算 y0 和 y1 任务对共享层第一层权重的拉扯梯度
    grad_y0 = torch.autograd.grad(loss_y0, first_layer_w, retain_graph=True)[0]
    grad_y1 = torch.autograd.grad(loss_y1, first_layer_w)[0]

    # --- 统计预估水位与梯度状态 ---
    with torch.no_grad():
        p0 = torch.sigmoid(y0)
        # NT 人群的 PCOC (找出水位下不去的原因)
        pcoc_nt = p0[pi_00 > 0.8].mean().item() / TRUE_AVG_CTR
        
        # 梯度余弦相似度 (核心)
        cos = F.cosine_similarity(grad_y0.view(1, -1), grad_y1.view(1, -1)).item()
        
    return {
        "cos": cos, 
        "pcoc_nt": pcoc_nt, 
        "y0_mean": p0.mean().item(),
        "head_bias": model.head_0.bias.mean().item(),
        "grad_norm_y0": grad_y0.norm().item(),
        "grad_norm_y1": grad_y1.norm().item()
    }

# ==========================================
# 3. 主程序：构建流水线
# ==========================================
def main():
    print("\n" + "="*70)
    print("🔬 TARNET 动力学对冲深度扫描 (V3 完美架构适配版)")
    print("="*70)

    # --- Step 1: 解析 Config ---
    print("⚙️ 正在解析 C 模型 Config...")
    with open(C_CONFIG_PATH, 'r') as f:
        config = json.load(f)
    h_dims = config.get('hidden_dims', [128, 64, 32])
    dropout = config.get('dropout_rate', 0.0)
    print(f"   - 捕获到网络结构: Hidden Dims = {h_dims}")

    # --- Step 2: 加载数据与因果先验 ---
    print("📦 正在加载测试数据...")
    limit = 4096
    df_x = pd.read_parquet(PARQUET_PATH).head(limit)
    x_cont = torch.tensor(df_x[[f"f{i}" for i in range(12)]].values, dtype=torch.float32)

    df_c = pd.read_csv(C_PRED_PATH).head(limit)
    c0_prob = torch.tensor(df_c['y0_prob'].values, dtype=torch.float32)
    c1_prob = torch.tensor(df_c['y1_prob'].values, dtype=torch.float32)
    # Rubin 概率推导
    pi_00 = (1.0 - c1_prob).clamp(min=0.0) 
    pi_01 = (c1_prob - c0_prob).clamp(min=0.0)

    # --- Step 3: 实例化并预热 C 模型 ---
    print("🧠 正在加载并冻结 Model C...")
    c_model = TARNET_Baseline(
        continuous_dim=12, categorical_cardinalities={},
        hidden_dims=h_dims, dropout_rate=dropout
    )
    c_ckpt = torch.load(C_MODEL_PATH, map_location=DEVICE)
    c_model.load_state_dict(c_ckpt.get('model_state_dict', c_ckpt), strict=False)
    # 彻底冻结 C 模型 (与你 models.py 里的逻辑契合)
    c_model.eval()
    c_model.requires_grad_(False)

    # --- Step 4: 实例化 V1 与 V2 模型 ---
    print("🏗️ 正在组装 V1 (Base) 与 V2 (Proposed) 网络...")
    model_v1 = TARNET_Baseline(
        continuous_dim=12, categorical_cardinalities={},
        hidden_dims=h_dims, dropout_rate=dropout
    )
    
    model_v2 = TARNET_Proposed(
        continuous_dim=12, categorical_cardinalities={},
        hidden_dims=[64,32], dropout_rate=dropout,
        c_fusion_mode="joint_emb", 
        c_embedding_dim=16, # 强制对齐你之前提到的 16 维
        c_model=c_model     # 注入刚刚加载好的 C 模型
    )

    # --- Step 5: 执行心电图诊断 ---
    print("🚀 正在执行前向与反向梯度探测...\n")
    res_v1 = get_grad_stats_robust(model_v1, V1_PATH, x_cont, pi_00, pi_01)
    res_v2 = get_grad_stats_robust(model_v2, V2_PATH, x_cont, pi_00, pi_01)

    # --- Step 6: 打印最终判决报表 ---
    print(f"{'诊断指标':<20} | {'V1 (无 Emb 纯 X)':<18} | {'V2 (拼接 16维 Emb)':<18}")
    print("-" * 65)
    print(f"{'梯度对齐度 (Cosine)':<20} | {res_v1['cos']:18.4f} | {res_v2['cos']:18.4f}")
    print(f"{'NT PCOC (大盘水位)':<20} | {res_v1['pcoc_nt']:18.4f} | {res_v2['pcoc_nt']:18.4f}")
    print(f"{'y0 梯度绝对能量':<20} | {res_v1['grad_norm_y0']:18.4f} | {res_v2['grad_norm_y0']:18.4f}")
    print(f"{'y1 梯度绝对能量':<20} | {res_v1['grad_norm_y1']:18.4f} | {res_v2['grad_norm_y1']:18.4f}")

    # Bias Oracle 实验
    print("\n" + "-"*65)
    offset = TARGET_LOGIT - res_v2['head_bias']
    new_logit = np.log(res_v2['y0_mean']/(1-res_v2['y0_mean'])) + offset
    new_pcoc = (torch.sigmoid(torch.tensor(new_logit))).item() / TRUE_AVG_CTR
    print(f"[Bias Oracle] 假设将 V2 y0 塔的 Bias 初始化为 {TARGET_LOGIT:.2f}:")
    print(f" 👉 预期 NT PCOC 会由 {res_v2['pcoc_nt']:.2f} 降至 {new_pcoc:.4f}")

if __name__ == "__main__":
    main()


🔬 TARNET 动力学对冲深度扫描 (V3 完美架构适配版)
⚙️ 正在解析 C 模型 Config...
   - 捕获到网络结构: Hidden Dims = [128, 64, 32]
📦 正在加载测试数据...
🧠 正在加载并冻结 Model C...
🏗️ 正在组装 V1 (Base) 与 V2 (Proposed) 网络...
🚀 正在执行前向与反向梯度探测...

诊断指标                 | V1 (无 Emb 纯 X)     | V2 (拼接 16维 Emb)   
-----------------------------------------------------------------
梯度对齐度 (Cosine)       |            -0.8991 |            -0.6813
NT PCOC (大盘水位)       |             0.3807 |             3.4987
y0 梯度绝对能量            |             0.0255 |             0.0590
y1 梯度绝对能量            |             0.0488 |             0.0727

-----------------------------------------------------------------
[Bias Oracle] 假设将 V2 y0 塔的 Bias 初始化为 -6.21:
 👉 预期 NT PCOC 会由 3.50 降至 0.0076
